# Phase Shift Calculation & THz Electric Field Derivation

Derives the THz electric field E_THz from the measured BPD voltage using the TFLN MZI sensor equations from:

> Wilke et al., *Thin-film lithium niobate electro-optic terahertz wave detector*, Scientific Reports 14:4822 (2024)

## Key equations

**Phase shift:**
$$\Delta\phi(\omega_{RF}) = \frac{\omega_{opt}}{c} T_{RF}(\omega_{RF})\, n_e^3\, r_{33}\, E_{THz}\, l$$

**Transfer function (velocity matching):**
$$T_{RF} = \frac{\sin\!\left(\frac{\omega_{RF}(n_{RF}-n_{opt})l}{2c}\right)}{\frac{\omega_{RF}(n_{RF}-n_{opt})l}{2c}}$$

**Half-wave field (dynamic range ceiling):**
$$E_\pi = \frac{\lambda}{2\,T_{RF}\,n_e^3\,r_{33}\,l}$$

So: $\Delta\phi = \pi\,E_{THz}/E_\pi$ and **$E_{THz} = (\Delta\phi/\pi)\,E_\pi$**

**BPD-to-phase:** At quadrature, $V_{BPD} = V_{max}\sin(\Delta\phi) \approx V_{max}\,\Delta\phi$, so $\Delta\phi = V_{signal}/V_{max}$.

In [1]:
import os
import matplotlib
import numpy as np
import matplotlib.pyplot as plt
from scipy.ndimage import uniform_filter1d

if os.path.basename(os.getcwd()) == 'Analysis':
    os.chdir('..')

os.makedirs('Results', exist_ok=True)

lam    = 1550e-9
n_e    = 2.15
r33    = 30.9e-12
l      = 600e-6
n_RF   = 1.95
n_opt  = 2.4
c      = 3e8
T_RF   = 1.0
E_pi   = lam / (2 * T_RF * n_e**3 * r33 * l)
transduction = np.pi / E_pi

print(f'E_pi  = {E_pi/1e6:.3f} MV/m  (paper reports 4.2 MV/m ✓)')
print(f'CWD: {os.getcwd()}')

RF_Hz      = 81.25e6
T_LASER_NS = 1e9 / RF_Hz
mV         = 1e3
SIGNAL_START_NS = 45.0
SIGNAL_END_NS   = 75.0
FLANK_END_NS    = 100.0
FFT_SIGMA_GHZ   = 0.35
FFT_ORDER       = 6
PROBE_HALF_WIN  = 25

E_pi  = 4.206 MV/m  (paper reports 4.2 MV/m ✓)
CWD: /home/paulrleaa/Documents/1_Research/Suli2026/SULI_THZ_SUM26


In [2]:
# ── T_RF(ω) as a function of frequency ───────────────────────────────────────
freqs_THz = np.linspace(0.001, 1.0, 1000)  # THz
omega_RF  = 2 * np.pi * freqs_THz * 1e12   # rad/s
arg       = omega_RF * (n_RF - n_opt) * l / (2 * c)
T_RF_f    = np.sinc(arg / np.pi)  # sinc(x) = sin(πx)/(πx) in numpy

fig, ax = plt.subplots(figsize=(7, 4))
ax.plot(freqs_THz * 1e3, T_RF_f**2, 'C0', lw=2)
ax.axhline(0.5, color='C3', ls='--', lw=1, label='-3 dB')
f3dB = freqs_THz[np.argmin(np.abs(T_RF_f**2 - 0.5))]
ax.axvline(f3dB*1e3, color='C3', ls=':', lw=1)
ax.set_xlabel('Frequency (GHz)', fontsize=11)
ax.set_ylabel('|T_RF|²', fontsize=11)
ax.set_title(f'TFLN MZI Transfer Function (l={l*1e6:.0f} μm, fused silica)\n3 dB bandwidth ≈ {f3dB*1e3:.0f} GHz')
ax.legend(); ax.grid(True, alpha=0.35)
ax.set_xlim(0, 1000)
plt.tight_layout()
plt.savefig('Results/T_RF_transfer_function.png', dpi=200, bbox_inches='tight')
plt.show()
print(f'3 dB bandwidth ≈ {f3dB*1e3:.0f} GHz')

3 dB bandwidth ≈ 492 GHz


/tmp/ipykernel_45695/1530284853.py:19: UserWarning: FigureCanvasAgg is non-interactive, and thus cannot be shown
  plt.show()


In [3]:
# ── EO signal extraction pipeline ────────────────────────────────────────────
def _period_fill_lookup(t_ns, s0, s1, fe):
    dt_ns  = (t_ns[-1] - t_ns[0]) / (len(t_ns) - 1)
    T_samp = T_LASER_NS / dt_ns
    i0 = int(round(s0 / dt_ns)); i1 = int(round(s1 / dt_ns))
    idx  = np.arange(i0, i1)
    n_l  = np.ceil((idx + 1 - i0) / T_samp).astype(int)
    n_r  = np.ceil((i1 - idx)     / T_samp).astype(int)
    return i0, i1, np.round(idx - n_l*T_samp).astype(int), np.round(idx + n_r*T_samp).astype(int)

def fft_lowpass(residual, dt_ns):
    n     = residual.shape[1]
    freqs = np.fft.rfftfreq(n, d=dt_ns)
    filt  = np.exp(-0.5 * (freqs / FFT_SIGMA_GHZ) ** (2 * FFT_ORDER))
    return np.fft.irfft(np.fft.rfft(residual, axis=1) * filt, n=n, axis=1)

def process_scan(path):
    d        = np.load(path)
    t_ns     = 1e9 * d['x'][0, :]
    phase    = d['phase']
    delay_ps = (phase - phase[0]) / RF_Hz / 360 * 1e12
    dt_ns    = (t_ns[-1] - t_ns[0]) / (len(t_ns) - 1)
    traces   = d['BPDwf'].mean(axis=1)  # (steps, 5000)

    i0, i1, sl, sr = _period_fill_lookup(t_ns, SIGNAL_START_NS, SIGNAL_END_NS, FLANK_END_NS)
    bl = traces.copy(); bl[:, i0:i1] = 0.5*(traces[:, sl] + traces[:, sr])
    res = fft_lowpass(traces - bl, dt_ns)

    t_probe  = res.var(axis=0).argmax()
    win      = slice(max(0, t_probe-PROBE_HALF_WIN), min(res.shape[1], t_probe+PROBE_HALF_WIN))
    sig_V    = res[:, win].mean(axis=1)   # signed amplitude in V
    V_max    = np.median(traces[:, t_probe])  # carrier level V
    return delay_ps, sig_V, V_max, t_ns, traces, t_probe

SCANS = [
    (0.25, 'Data_raw/July31_/finescaneQ0p25nC_wh5_4.npz'),
    (0.50, 'Data_raw/July31_/finescaneQ0p5nC_wh5_1.npz'),
    (1.00, 'Data_raw/July31_/finescaneQ1nC_wh5_6.npz'),
    (4.00, 'Data_raw/July31_/finescaneQ4nC_wh4_5.npz'),
    (5.00, 'Data_raw/July31_/finescaneQ5nC_wh4_3.npz'),
    (7.00, 'Data_raw/July31_/finescaneQ7nC_wh4_2.npz'),
    (10.0, 'Data_raw/July31_/finescaneQ10nC_wh4_0.npz'),
]

scan_data = []
for Q_nom, path in SCANS:
    dp, sig, vmax, t_ns, traces, tp = process_scan(path)
    delta_phi = sig / vmax                          # rad (signed)
    E_THz_Vm  = (delta_phi / np.pi) * E_pi          # V/m (signed)
    peak_phi  = np.abs(delta_phi).max()
    peak_E    = np.abs(E_THz_Vm).max()
    scan_data.append({'Q': Q_nom, 'delay_ps': dp, 'sig_V': sig,
                      'delta_phi': delta_phi, 'E_THz': E_THz_Vm,
                      'V_max': vmax, 'peak_phi': peak_phi, 'peak_E': peak_E})
    print(f'{Q_nom:5.2f} nC | V_max={vmax*1e3:6.1f} mV | peak Δφ={peak_phi*1e3:6.1f} mrad | peak E={peak_E/1e6:.3f} MV/m')

 0.25 nC | V_max= 174.0 mV | peak Δφ=  71.2 mrad | peak E=0.095 MV/m
 0.50 nC | V_max= 179.2 mV | peak Δφ= 141.3 mrad | peak E=0.189 MV/m


 1.00 nC | V_max= 175.2 mV | peak Δφ= 257.4 mrad | peak E=0.345 MV/m


 4.00 nC | V_max= 172.2 mV | peak Δφ= 745.6 mrad | peak E=0.998 MV/m
 5.00 nC | V_max= 170.0 mV | peak Δφ= 985.0 mrad | peak E=1.319 MV/m


 7.00 nC | V_max= 174.0 mV | peak Δφ=1185.9 mrad | peak E=1.588 MV/m


10.00 nC | V_max= 168.0 mV | peak Δφ=1442.6 mrad | peak E=1.931 MV/m


## Corrected nonlinear phase extraction

The linear formula $\Delta\phi = V_{signal}/V_{carrier}$ has two issues:

1. **Wrong normalisation**: the carrier $V_{carrier} = V_{max}\sin(\phi_{bias})$ is used as the denominator, but the correct denominator for phase is the *sensitivity* $V_{max}\cos(\phi_{bias})$.  These are equal only when $\phi_{bias} = 45°$ — which happens to hold approximately here, so the error from this is small.

2. **Small-angle breakdown**: $\sin(\Delta\phi) \approx \Delta\phi$ fails for large phase shifts.  At 4 nC the error is ~10%; at 10 nC it reaches ~31%.  The MZI is being driven far into its nonlinear regime.

**Corrected approach:**

$$V_{max} = \frac{V_{probe,max} - V_{probe,min}}{2}, \qquad \phi_{bias} = \arcsin\!\left(\frac{V_{carrier}}{V_{max}}\right)$$

$$\Delta\phi_{exact} = \arcsin\!\left(\frac{V_{signal}}{V_{max}} + \sin\phi_{bias}\right) - \phi_{bias}$$

When $V_{signal}/V_{max} + \sin\phi_{bias} > 1$ the argument of arcsin exceeds 1 — the BPD has been driven past the peak of its transfer curve and the phase cannot be recovered without additional information (labelled **CLIPPED** below).

In [4]:
def estimate_vmax_and_bias(scans):
    """
    Estimate V_max and φ_bias from ALL scans together.

    V_max (the max BPD output at the probe time if fully modulated) is
    determined by laser power × detector gain — it should be constant
    across all scans since the laser doesn't change with beam charge.

    At low charge the EO signal barely swings the probe, so (max-min)/2
    at the probe time gives a nearly-zero V_max that is meaningless.
    At high charge the BPD sweeps a large arc, giving a reliable V_max.

    Strategy: use the union of all probe-time values across all scans to
    estimate V_max from the widest observed swing.
    """
    from scipy.ndimage import uniform_filter1d

    all_probe_vals = []
    ref_path = scans[-1][1]   # 10 nC — highest SNR, widest swing
    d        = np.load(ref_path)
    t_ns     = 1e9 * d['x'][0]
    dt_ns    = (t_ns[-1]-t_ns[0])/(len(t_ns)-1)
    traces   = d['BPDwf'].mean(axis=1)

    # Period-fill and FFT to find probe time from 10 nC scan
    i0,i1,sl,sr = _period_fill_lookup(t_ns, SIGNAL_START_NS, SIGNAL_END_NS, FLANK_END_NS)
    bl = traces.copy(); bl[:,i0:i1] = 0.5*(traces[:,sl]+traces[:,sr])
    res = fft_lowpass(traces-bl, dt_ns)
    t_probe = res.var(axis=0).argmax()

    # Collect probe-time values from all scans
    for _, path in scans:
        d2 = np.load(path)
        tr = d2['BPDwf'].mean(axis=1)
        all_probe_vals.append(tr[:, t_probe])

    all_pv  = np.concatenate(all_probe_vals)
    V_max   = (all_pv.max() - all_pv.min()) / 2
    V_carrier = np.median(all_probe_vals[-1])   # 10 nC carrier as reference

    # Bias angle from 10 nC (most reliable)
    sin_bias = np.clip(V_carrier / V_max, -1, 1)
    phi_bias = np.arcsin(sin_bias)

    return V_max, phi_bias, t_probe


def correct_phase(path, E_pi_val, V_max_shared, phi_bias_shared, t_probe_shared):
    """
    Returns linear and nonlinear corrected phase for a scan.

    V_max_shared and phi_bias_shared are estimated once from the full
    dataset (not per-scan) to avoid division by tiny probe swings at low charge.
    """
    d        = np.load(path)
    t_ns     = 1e9 * d['x'][0]
    phase    = d['phase']
    delay_ps = (phase - phase[0]) / RF_Hz / 360 * 1e12
    dt_ns    = (t_ns[-1] - t_ns[0]) / (len(t_ns) - 1)
    traces   = d['BPDwf'].mean(axis=1)

    i0,i1,sl,sr = _period_fill_lookup(t_ns, SIGNAL_START_NS, SIGNAL_END_NS, FLANK_END_NS)
    bl = traces.copy(); bl[:,i0:i1] = 0.5*(traces[:,sl]+traces[:,sr])
    res = fft_lowpass(traces-bl, dt_ns)

    win   = slice(max(0, t_probe_shared-PROBE_HALF_WIN),
                  min(res.shape[1], t_probe_shared+PROBE_HALF_WIN))
    sig_V = res[:, win].mean(axis=1)

    V_carrier = np.median(traces[:, t_probe_shared])

    # ── Linear (original) ────────────────────────────────────────────────
    dph_lin = sig_V / V_carrier
    E_lin   = (dph_lin / np.pi) * E_pi_val

    # ── Corrected (exact arcsin) ──────────────────────────────────────────
    sin_bias = np.sin(phi_bias_shared)
    arg      = sig_V / V_max_shared + sin_bias
    clipped  = np.abs(arg) > 1
    dph_cor  = np.arcsin(np.clip(arg, -1, 1)) - phi_bias_shared
    E_cor    = (dph_cor / np.pi) * E_pi_val
    dph_cor[clipped] = np.nan
    E_cor[clipped]   = np.nan

    return {
        'delay_ps': delay_ps, 'sig_V': sig_V,
        'dph_lin': dph_lin, 'E_lin': E_lin,
        'dph_cor': dph_cor, 'E_cor': E_cor,
        'V_carrier': V_carrier, 'clipped': clipped,
        'clip_frac': clipped.mean(),
    }


# ── Estimate shared V_max and φ_bias from all scans ──────────────────────
V_max_shared, phi_bias_shared, t_probe_shared = estimate_vmax_and_bias(SCANS)
print(f'Shared V_max   = {V_max_shared*1e3:.1f} mV')
print(f'Shared φ_bias  = {np.degrees(phi_bias_shared):.1f}°')
print(f'Probe time     = {1e9*np.load(SCANS[0][1])["x"][0,t_probe_shared]:.1f} ns (sample {t_probe_shared})')
print()

# ── Run correction for all charges ───────────────────────────────────────
corrected = []
print(f"{'Q':>5} | {'V_carrier':>10} | {'peak Δφ_lin':>12} | {'peak Δφ_cor':>12} | {'over-est':>9} | {'clipped%':>9}")
print('-' * 75)
for Q_nom, path in SCANS:
    r = correct_phase(path, E_pi, V_max_shared, phi_bias_shared, t_probe_shared)
    r['Q'] = Q_nom
    corrected.append(r)
    pk_lin = np.nanmax(np.abs(r['dph_lin']))
    pk_cor = np.nanmax(np.abs(r['dph_cor']))
    over   = 100*(pk_lin/pk_cor - 1) if not np.isnan(pk_cor) else float('nan')
    print(f"{Q_nom:5.2f} | {r['V_carrier']*1e3:9.1f}mV | "
          f"{pk_lin*1e3:11.1f}mrad | {pk_cor*1e3 if not np.isnan(pk_cor) else float('nan'):11.1f}mrad | "
          f"{over:8.1f}% | {100*r['clip_frac']:8.1f}%")

Shared V_max   = 244.4 mV
Shared φ_bias  = 43.4°
Probe time     = 52.5 ns (sample 2624)

    Q |  V_carrier |  peak Δφ_lin |  peak Δφ_cor |  over-est |  clipped%
---------------------------------------------------------------------------
 0.25 |     173.0mV |        72.0mrad |        68.1mrad |      5.8% |      0.0%
 0.50 |     172.4mV |       144.3mrad |       151.6mrad |     -4.8% |      0.0%


 1.00 |     173.6mV |       256.9mrad |       297.0mrad |    -13.5% |      0.0%
 4.00 |     165.6mV |       761.4mrad |       471.4mrad |     61.5% |      1.9%
 5.00 |     140.4mV |      1145.0mrad |       728.3mrad |     57.2% |      2.5%


 7.00 |     162.0mV |      1264.0mrad |       908.9mrad |     39.1% |      3.8%
10.00 |     168.0mV |      1442.6mrad |      1063.5mrad |     35.6% |      5.0%


In [5]:
colors = plt.cm.plasma(np.linspace(0.15, 0.85, len(corrected)))

fig, axes = plt.subplots(1, 3, figsize=(17, 5))

# ── Panel 1: linear vs corrected E-field waveforms ───────────────────────
ax = axes[0]
for r, col in zip(corrected, colors):
    dp = r['delay_ps']
    ax.plot(dp, r['E_lin']/1e6, color=col, lw=1.2, ls='--', alpha=0.6)
    ax.plot(dp, r['E_cor']/1e6, color=col, lw=1.8,
            label=f"{r['Q']:.2g} nC")
    # Mark clipped steps
    if r['clipped'].any():
        cl_dp = dp[r['clipped']]
        ax.scatter(cl_dp, np.zeros_like(cl_dp), color=col,
                   marker='x', s=25, zorder=5, linewidths=1)
ax.axhline(0, color='k', lw=0.6, ls='--', alpha=0.4)
ax.set_xlabel('Phase delay (ps)', fontsize=11)
ax.set_ylabel('E_THz (MV/m)', fontsize=11)
ax.set_title('E-field waveforms\n— dashed = linear (original)   solid = corrected\n× = clipped steps (invalid)')
ax.legend(title='Charge', fontsize=8); ax.grid(True, alpha=0.3)

# ── Panel 2: peak E-field comparison ─────────────────────────────────────
ax = axes[1]
Q_arr    = np.array([r['Q'] for r in corrected])
E_lin_pk = np.array([np.nanmax(np.abs(r['E_lin'])) for r in corrected])
E_cor_pk = np.array([np.nanmax(np.abs(r['E_cor'])) for r in corrected])
valid    = ~np.isnan(E_cor_pk)

ax.scatter(Q_arr, E_lin_pk/1e6, marker='s', s=60, color='C7', zorder=5,
           label='Linear (original)', alpha=0.8)
ax.scatter(Q_arr[valid], E_cor_pk[valid]/1e6, marker='o', s=70,
           c=[colors[i] for i in range(len(Q_arr)) if valid[i]], zorder=6,
           label='Corrected (arcsin)')
for i, (Q, ec, el) in enumerate(zip(Q_arr, E_cor_pk, E_lin_pk)):
    if not np.isnan(ec):
        over = 100*(el-ec)/ec
        ax.annotate(f'+{over:.0f}%', (Q, el/1e6), xytext=(3, 5),
                    textcoords='offset points', fontsize=8, color='C7')
    else:
        ax.annotate('CLIPPED', (Q, el/1e6), xytext=(3, 5),
                    textcoords='offset points', fontsize=8, color='C3')

ax.set_xlabel('Nominal charge (nC)', fontsize=11)
ax.set_ylabel('Peak |E_THz| (MV/m)', fontsize=11)
ax.set_title('Peak E-field: linear vs corrected\n(% = linear over-estimate)')
ax.legend(fontsize=9); ax.grid(True, alpha=0.3)
ax.set_xlim(left=0); ax.set_ylim(bottom=0)

# ── Panel 3: fraction of clipped steps per scan ───────────────────────────
ax = axes[2]
clip_fracs = [100*r['clip_frac'] for r in corrected]
bars = ax.bar(range(len(corrected)), clip_fracs,
              color=[colors[i] for i in range(len(corrected))], alpha=0.85)
ax.set_xticks(range(len(corrected)))
ax.set_xticklabels([f"{r['Q']:.2g} nC" for r in corrected],
                   rotation=30, ha='right', fontsize=9)
ax.set_ylabel('Fraction of steps where BPD is clipped (%)')
ax.set_title('Clipping fraction per scan\n(arcsin argument > 1: phase unrecoverable)')
ax.axhline(0, color='k', lw=0.5)
ax.grid(True, alpha=0.3, axis='y')
for bar, frac in zip(bars, clip_fracs):
    if frac > 0:
        ax.text(bar.get_x()+bar.get_width()/2, frac+0.5,
                f'{frac:.0f}%', ha='center', fontsize=9)

plt.suptitle('Nonlinear phase correction — exact arcsin vs linear approximation',
             fontsize=12, y=1.01)
plt.tight_layout()
plt.savefig('Results/phase_correction_comparison.png', dpi=200, bbox_inches='tight')
plt.show()

# ── Updated E_THz scaling using corrected values ──────────────────────────
print('\n=== E-field scaling with corrected phase ===')
Q_valid   = Q_arr[valid]
E_valid   = E_cor_pk[valid]
slope_cor = (Q_valid * E_valid).sum() / (Q_valid**2).sum()
print(f'Valid charges (not clipped): {Q_valid} nC')
print(f'Corrected beamline factor: {slope_cor/1e3:.1f} kV/m per nC')
print(f'(Previously linear: ~217 kV/m per nC — corrected is larger because arcsin > linear)')


=== E-field scaling with corrected phase ===
Valid charges (not clipped): [ 0.25  0.5   1.    4.    5.    7.   10.  ] nC
Corrected beamline factor: 160.4 kV/m per nC
(Previously linear: ~217 kV/m per nC — corrected is larger because arcsin > linear)


/tmp/ipykernel_45695/3284956623.py:71: UserWarning: FigureCanvasAgg is non-interactive, and thus cannot be shown
  plt.show()


## Proper MZI calibration from the charge series

The previous "corrected" approach has a hidden flaw: `V_max = (probe-time swing)/2` does NOT equal the MZI half-amplitude `V_0`. The observed swing is

$$\frac{V_{BPD,max} - V_{BPD,min}}{2} = V_0 \cos\phi_{bias}\,\sin\Delta\phi_{peak}$$

so the estimate is contaminated by `cos(φ_bias)·sin(Δφ_peak)` — the very quantities we want to measure. `φ_bias` then comes back as `arcsin(V_carrier/V_max_estimated)`, making the result self-referential rather than calibrated.

**The system is underdetermined per scan**: 2 observables (`V_carrier`, swing/2), 3 unknowns (`V_0`, `φ_bias`, `Δφ_peak`).

### Calibration strategy

No bias-sweep file exists in `Data_raw/`, but we have two independent constraints:

1. **Multi-charge linearity**: the THz field at the AWA scales linearly with bunch charge in the operating range (no saturation up to ~10 nC). So `Δφ_peak(Q) = α·Q` for some constant `α`. The full BPD trace at `t_probe` then satisfies
   $$V_{BPD,max}(Q) = V_0\sin(\phi_{bias} + \alpha_+ Q) + V_{offset}$$
   $$V_{BPD,min}(Q) = V_0\sin(\phi_{bias} - \alpha_- Q) + V_{offset}$$
   With 7 charges × {max, min} = 14 data points and 4 free parameters `(V_0, φ_bias, α_+, α_-)`.

2. **`V_offset` from trace minima**: between laser pulses (the zeros of the `|sin(2πf_laser t)|` envelope) the MZI output is gated off, so `V_BPD ≈ V_offset` — the BPD's electronic DC offset. Estimated from `min(trace)` per step.

3. **Cross-check from dipoleON (THz-off) scan**: with the dipole magnet steering the beam away, `Δφ = 0` and the BPD at `t_probe` reads `V_carrier = V_0 sin(φ_bias) + V_offset` directly. This gives us an independent measurement to validate the fit.

### Additional fixes in this section
- **Per-scan `t_probe`** instead of using the 10 nC value for all charges (laser timing can drift between runs).
- **Exact arcsin phase inversion**: `Δφ = arcsin((V_BPD − V_offset)/V_0) − φ_bias`, clipping anything that lands outside [−1, 1].
- **T_RF(ω) deconvolution** in the frequency domain to recover the true `E_THz(t)` instead of the DC-T_RF-assumed apparent field.

In [ ]:
# ── MZI calibration from the charge series ─────────────────────────────────
from scipy.optimize import curve_fit

# DipoleON file = dipole magnet energized → beam steered AWAY from crystal → no THz
DIPOLE_OFF_PATH = 'Data_raw/July31_/cosrsescaneQ10nC_wh4_0_dipoleON.npz'

def per_scan_probe(path):
    """(t_ns, traces, residual, t_probe, dt_ns) using per-scan max-variance sample."""
    d      = np.load(path)
    t_ns   = 1e9 * d['x'][0]
    dt_ns  = (t_ns[-1]-t_ns[0])/(len(t_ns)-1)
    traces = d['BPDwf'].mean(axis=1)
    i0,i1,sl,sr = _period_fill_lookup(t_ns, SIGNAL_START_NS, SIGNAL_END_NS, FLANK_END_NS)
    bl = traces.copy(); bl[:,i0:i1] = 0.5*(traces[:,sl]+traces[:,sr])
    res = fft_lowpass(traces - bl, dt_ns)
    t_probe = res.var(axis=0).argmax()
    return t_ns, traces, res, t_probe, dt_ns

# ── Step 1: per-scan probe-time stats + per-scan V_offset ──────────────────
cal_rows = []
for Q_nom, path in SCANS:
    _, traces, _, t_probe, _ = per_scan_probe(path)
    pv = traces[:, t_probe]
    # V_offset: laser-pulse off-time (between |sin| envelope peaks) → MZI gated off,
    # BPD output ≈ electronic DC offset only. Median of per-step trace minima.
    V_offset_scan = np.median(traces.min(axis=1))
    cal_rows.append({'Q': Q_nom,
                     'V_min':     pv.min(),
                     'V_carrier': np.median(pv),
                     'V_max':     pv.max(),
                     'V_offset':  V_offset_scan,
                     't_probe':   t_probe,
                     'path':      path})

Q_cal  = np.array([r['Q']         for r in cal_rows])
Vmax_c = np.array([r['V_max']     for r in cal_rows])
Vmin_c = np.array([r['V_min']     for r in cal_rows])
Vcar_c = np.array([r['V_carrier'] for r in cal_rows])
Voff_c = np.array([r['V_offset']  for r in cal_rows])

# Subtract per-scan V_offset → "MZI-only" amplitudes
Vmax_n = Vmax_c - Voff_c
Vmin_n = Vmin_c - Voff_c
Vcar_n = Vcar_c - Voff_c

# ── Step 2: dipoleON (THz off) → independent V_carrier reference ───────────
_, tr_off, _, tp_off, _ = per_scan_probe(DIPOLE_OFF_PATH)
V_carrier_ref = np.median(tr_off[:, tp_off])
V_off_ref     = np.median(tr_off.min(axis=1))
V_carrier_ref_n = V_carrier_ref - V_off_ref   # = V_0·sin(φ_bias)
print(f'DipoleON (THz off) reference:')
print(f'  V_carrier            = {V_carrier_ref*1e3:.1f} mV')
print(f'  V_offset             = {V_off_ref*1e3:.1f} mV')
print(f'  → V_0·sin(φ_bias)    = {V_carrier_ref_n*1e3:.1f} mV')

# ── Step 3: joint fit of (V_0, φ_bias, α_+, α_-) ──────────────────────────
def mzi_model(Q_signed, V_0, phi_bias, alpha_p, alpha_n):
    """Pos-peak (Q>0) uses α_+, neg-peak (Q<0) uses α_-. THz pulse is asymmetric."""
    is_pos = Q_signed > 0
    alpha  = np.where(is_pos, alpha_p, alpha_n)
    sign   = np.where(is_pos, +1.0, -1.0)
    return V_0 * np.sin(phi_bias + sign * alpha * np.abs(Q_signed))

# Reasonable initial conditions
V_0_init     = 0.35
phi_init     = float(np.arctan2(V_carrier_ref_n, (Vmax_n[-1]-Vmin_n[-1])/2))
alpha_p_init = 0.10
alpha_n_init = 0.07

Q_signed = np.concatenate([ Q_cal, -Q_cal])
V_stack  = np.concatenate([Vmax_n,  Vmin_n])

p_opt, p_cov = curve_fit(
    mzi_model, Q_signed, V_stack,
    p0=[V_0_init, phi_init, alpha_p_init, alpha_n_init],
    bounds=([0.05, 0.01, 0.001, 0.001], [2.0, np.pi/2-0.01, 1.0, 1.0]),
    maxfev=30000,
)
V_0_fit, phi_bias_fit, alpha_p_fit, alpha_n_fit = p_opt
p_err = np.sqrt(np.diag(p_cov))

print(f'\nJoint MZI fit (split α for asymmetric THz pulse):')
print(f'  V_0       = {V_0_fit*1e3:6.1f} ± {p_err[0]*1e3:.1f} mV')
print(f'  φ_bias    = {np.degrees(phi_bias_fit):6.2f} ± {np.degrees(p_err[1]):.2f}°  '
      f'({phi_bias_fit:.3f} rad)')
print(f'  α_+ (pos) = {alpha_p_fit:6.4f} ± {p_err[2]:.4f} rad/nC')
print(f'  α_- (neg) = {alpha_n_fit:6.4f} ± {p_err[3]:.4f} rad/nC')
print(f'  asymmetry α_+/α_- = {alpha_p_fit/alpha_n_fit:.2f}')

V_pred = mzi_model(Q_signed, *p_opt)
print(f'  RMS residual: {(V_stack-V_pred).std()*1e3:.2f} mV  '
      f'(carrier-excess varies {Vcar_n.std()*1e3:.1f} mV across scans)')

# Cross-check against dipoleON
V_car_pred = V_0_fit * np.sin(phi_bias_fit)
print(f'\nCross-check V_0·sin(φ_bias) at Q=0:')
print(f'  Fit       = {V_car_pred*1e3:.1f} mV')
print(f'  DipoleON  = {V_carrier_ref_n*1e3:.1f} mV  (Δ = {(V_car_pred-V_carrier_ref_n)*1e3:+.1f} mV)')

# Sensitivity & comparison to the old "linear" formula's denominator
sens           = V_0_fit * np.cos(phi_bias_fit)
V_car_full_typ = float(np.median(Vcar_c))
over_factor    = sens / V_car_full_typ      # how much old Δφ_lin over-estimates true Δφ
print(f'\nLinear-formula error analysis (small-Δφ regime):')
print(f'  True sensitivity V_0·cos(φ_bias) = {sens*1e3:.1f} mV/rad')
print(f'  Old denom V_carrier (full)       = {V_car_full_typ*1e3:.1f} mV/rad')
print(f'  → old V_signal/V_carrier OVER-estimates true Δφ by factor {over_factor:.2f}×')
print(f'    (at large Δφ the arcsin nonlinearity partly cancels this — net error is smaller)')

# Store calibration for downstream use
MZI_CAL = dict(V_0=V_0_fit, phi_bias=phi_bias_fit,
               alpha_p=alpha_p_fit, alpha_n=alpha_n_fit,
               V_offset_median=float(np.median(Voff_c)))

# ── Step 4: plot the fit ───────────────────────────────────────────────────
fig, axes = plt.subplots(1, 2, figsize=(13, 5))

ax = axes[0]
Q_fine_p = np.linspace(0, 12, 200)
Q_fine_n = np.linspace(0, 12, 200)
ax.plot(Q_fine_p,    (V_0_fit*np.sin(phi_bias_fit + alpha_p_fit*Q_fine_p))*1e3,
        'C3-', lw=1.5, alpha=0.7, label=f'V_max model (α_+={alpha_p_fit:.3f} rad/nC)')
ax.plot(-Q_fine_n,   (V_0_fit*np.sin(phi_bias_fit - alpha_n_fit*Q_fine_n))*1e3,
        'C0-', lw=1.5, alpha=0.7, label=f'V_min model (α_-={alpha_n_fit:.3f} rad/nC)')
ax.scatter( Q_cal, Vmax_n*1e3, marker='^', s=70, color='C3', zorder=5, label='V_max − V_offset')
ax.scatter(-Q_cal, Vmin_n*1e3, marker='v', s=70, color='C0', zorder=5, label='V_min − V_offset')
ax.axhline(V_carrier_ref_n*1e3, color='C2', ls='--', lw=1,
           label=f'DipoleON V_carrier − V_off = {V_carrier_ref_n*1e3:.0f} mV')
ax.axhline(0, color='k', lw=0.6, ls=':', alpha=0.4)
ax.set_xlabel('Signed bunch charge (nC)  (+Q = THz pos-peak,  −Q = THz neg-peak)', fontsize=10)
ax.set_ylabel('V_BPD(t_probe) − V_offset  (mV)', fontsize=11)
ax.set_title('MZI calibration: V_BPD traces out the sin curve as Q scales Δφ')
ax.legend(fontsize=8, loc='upper left'); ax.grid(True, alpha=0.35)

ax = axes[1]
phi_view = np.linspace(-np.pi/2, np.pi, 500)
ax.plot(np.degrees(phi_view), (V_0_fit*np.sin(phi_view))*1e3, 'k-', lw=1.5, alpha=0.5,
        label='Full MZI sin curve (V_offset removed)')
ax.scatter([np.degrees(phi_bias_fit)], [V_0_fit*np.sin(phi_bias_fit)*1e3],
           s=120, color='C2', zorder=5, marker='o',
           label=f'Bias φ = {np.degrees(phi_bias_fit):.1f}°')
for Q_i, Vmx_n, Vmn_n in zip(Q_cal, Vmax_n, Vmin_n):
    phi_pos = phi_bias_fit + alpha_p_fit*Q_i
    phi_neg = phi_bias_fit - alpha_n_fit*Q_i
    ax.scatter([np.degrees(phi_pos)], [Vmx_n*1e3], color='C3', s=30, marker='^', zorder=4)
    ax.scatter([np.degrees(phi_neg)], [Vmn_n*1e3], color='C0', s=30, marker='v', zorder=4)
    ax.plot([np.degrees(phi_neg), np.degrees(phi_pos)],
            [Vmn_n*1e3, Vmx_n*1e3], 'C7-', lw=0.5, alpha=0.4)
ax.axvline(90, color='C3', ls=':', lw=1, alpha=0.4, label='quadrature (90°)')
ax.axhline(0, color='k', lw=0.6, ls='-', alpha=0.4)
ax.set_xlabel('Total MZI phase φ_bias + Δφ_signal  (deg)', fontsize=11)
ax.set_ylabel('V_BPD − V_offset  (mV)', fontsize=11)
ax.set_title('Operating region on the MZI transfer curve\n'
             '▲ V_max excursions  ▼ V_min excursions')
ax.legend(fontsize=8); ax.grid(True, alpha=0.35)

plt.tight_layout()
plt.savefig('Results/mzi_calibration_fit.png', dpi=200, bbox_inches='tight')
plt.show()

In [ ]:
# ── Calibrated phase extraction with per-scan t_probe ──────────────────────
def extract_phase_calibrated(path, cal):
    """
    Returns Δφ(delay) using the fitted MZI calibration.

    Per-scan steps:
      1. find t_probe (max-variance sample of the FFT-cleaned residual)
      2. compute V_signal = mean(res over PROBE_HALF_WIN samples)
      3. V_carrier(scan) = median(traces[:, t_probe])
      4. V_offset(scan)  = median(traces.min(axis=1))
      5. Δφ = arcsin( (V_BPD - V_offset) / V_0 ) - φ_bias
              = arcsin( V_signal/V_0 + (V_carrier-V_offset)/V_0 ) - φ_bias
         clip arg ∈ [-1, 1] and mark clipped points NaN.
    """
    V_0       = cal['V_0']
    phi_bias  = cal['phi_bias']

    t_ns, traces, res, t_probe, dt_ns = per_scan_probe(path)
    d         = np.load(path)
    phase     = d['phase']
    delay_ps  = (phase - phase[0]) / RF_Hz / 360 * 1e12

    win       = slice(max(0, t_probe-PROBE_HALF_WIN),
                      min(res.shape[1], t_probe+PROBE_HALF_WIN))
    V_signal  = res[:, win].mean(axis=1)                  # signed, baseline-subtracted
    V_carrier = float(np.median(traces[:, t_probe]))      # full V_carrier (incl. V_offset)
    V_offset  = float(np.median(traces.min(axis=1)))

    # V_BPD - V_offset = V_signal + (V_carrier - V_offset)
    arg       = V_signal / V_0 + (V_carrier - V_offset) / V_0
    clipped   = np.abs(arg) > 1.0
    dphi      = np.arcsin(np.clip(arg, -1, 1)) - phi_bias
    dphi[clipped] = np.nan

    # Old linear formula for comparison
    dphi_lin  = V_signal / V_carrier

    return {
        'delay_ps':  delay_ps,
        'V_signal':  V_signal,
        'V_carrier': V_carrier,
        'V_offset':  V_offset,
        't_probe':   t_probe,
        't_probe_ns': t_ns[t_probe],
        'dphi':      dphi,
        'dphi_lin':  dphi_lin,
        'clipped':   clipped,
        'clip_frac': clipped.mean(),
    }

calibrated = []
print(f"{'Q':>5} | {'t_probe(ns)':>11} | {'V_car(mV)':>9} | {'V_off(mV)':>9} | "
      f"{'peak |Δφ+|':>10} | {'peak |Δφ-|':>10} | {'clipped%':>8}")
print('-'*92)
for Q_nom, path in SCANS:
    r = extract_phase_calibrated(path, MZI_CAL)
    r['Q'] = Q_nom
    calibrated.append(r)
    pk_pos = np.nanmax(r['dphi'][r['dphi'] > 0]) if np.any(r['dphi'] > 0) else float('nan')
    pk_neg = np.abs(np.nanmin(r['dphi'][r['dphi'] < 0])) if np.any(r['dphi'] < 0) else float('nan')
    print(f"{Q_nom:5.2f} | {r['t_probe_ns']:11.2f} | "
          f"{r['V_carrier']*1e3:9.1f} | {r['V_offset']*1e3:9.1f} | "
          f"{pk_pos*1e3:9.1f}mrad | {pk_neg*1e3:9.1f}mrad | {100*r['clip_frac']:7.1f}%")

In [ ]:
# ── T_RF(ω) deconvolution: recover true E_THz(t) from Δφ(t) ────────────────
#
# Paper Eq.(1) is frequency-dependent:
#   Δφ̃(ω) = (2π/λ) T_RF(ω) n_e³ r₃₃ l · Ẽ_THz(ω)
# i.e. each frequency component of E_THz is attenuated by T_RF(ω) before producing Δφ.
# To recover E_THz(t) from the measured Δφ(t) we deconvolve in frequency:
#   Ẽ_THz(ω) = Δφ̃(ω) · E_π,DC / (π · T_RF(ω))
# where E_π,DC = λ/(2 n_e³ r₃₃ l) is the DC half-wave field (T_RF=1).
#
# T_RF(ω) → 0 around 1 THz, so naive 1/T_RF amplifies noise. We use a
# Wiener-style regulariser:
#   1/T_RF_reg(ω) ≡ T_RF(ω) / (T_RF(ω)² + ε²)
# Components where |T_RF| ≪ ε are smoothly suppressed instead of blowing up.

E_pi_DC = lam / (2 * 1.0 * n_e**3 * r33 * l)   # T_RF=1 half-wave field

def T_RF_of_omega(omega):
    arg = omega * (n_RF - n_opt) * l / (2 * c)
    return np.sinc(arg / np.pi)                 # sin(x)/x

def deconvolve_E_from_dphi(delay_ps, dphi, eps=0.10):
    """
    Convert Δφ(delay) → E_THz(delay) with frequency-domain T_RF deconvolution.

    delay_ps : (N,) uniformly spaced delays in ps
    dphi     : (N,) phase shift in rad (NaNs from clipping are interpolated first)
    eps      : Wiener cutoff on |T_RF|. Components with |T_RF| ≲ eps are damped.

    Returns
    -------
    E_nodeconv : "apparent" E-field assuming T_RF=1 everywhere (= (Δφ/π)·E_π,DC)
    E_deconv   : T_RF-deconvolved E_THz(t)
    f_GHz, T_RF_used, weight : diagnostics for the frequency axis
    """
    nan = np.isnan(dphi)
    if nan.any():
        dphi = np.array(dphi, copy=True)
        idx  = np.arange(len(dphi))
        dphi[nan] = np.interp(idx[nan], idx[~nan], dphi[~nan])

    N      = len(dphi)
    dt_s   = (delay_ps[-1] - delay_ps[0]) / (N - 1) * 1e-12
    win    = np.hanning(N)
    DPHI   = np.fft.rfft(dphi * win)
    omega  = 2 * np.pi * np.fft.rfftfreq(N, d=dt_s)
    T      = T_RF_of_omega(omega)
    weight = T / (T*T + eps*eps)                # regularised 1/T

    # Apparent E-field (T_RF=1)
    E_nodeconv = (dphi / np.pi) * E_pi_DC
    # Deconvolved: scale each freq component, then iFFT and un-window
    E_DEC      = (DPHI / np.pi) * E_pi_DC * weight
    E_deconv_w = np.fft.irfft(E_DEC, n=N)
    safe       = win > 0.02
    E_deconv   = np.where(safe, E_deconv_w / np.where(safe, win, 1.0), np.nan)
    return E_nodeconv, E_deconv, omega/(2*np.pi*1e9), T, weight

# ── Apply deconvolution to each calibrated scan ────────────────────────────
for r in calibrated:
    E_nd, E_dc, f_GHz, T_used, w_used = deconvolve_E_from_dphi(
        r['delay_ps'], r['dphi'], eps=0.10)
    r['E_no_deconv'] = E_nd
    r['E_deconv']    = E_dc
    r['f_GHz']       = f_GHz
    r['T_RF_used']   = T_used
    r['deconv_w']    = w_used

# ── Diagnostic plot: T_RF and the Wiener weight ────────────────────────────
fig, ax = plt.subplots(figsize=(8, 4.5))
ax.plot(f_GHz, T_used,                'C0-',  lw=1.5, label='T_RF(ω)')
ax.plot(f_GHz, T_used**2,             'C0--', lw=1.0, alpha=0.6, label='|T_RF|²')
ax.plot(f_GHz, w_used / w_used.max(), 'C3-',  lw=1.5, label='Wiener weight (norm.)')
ax.axhline(0.1, color='k', ls=':', lw=1, alpha=0.5, label='ε = 0.10')
ax.set_xlim(0, 1500); ax.set_ylim(-0.05, 1.05)
ax.set_xlabel('Frequency (GHz)', fontsize=11)
ax.set_ylabel('amplitude', fontsize=11)
ax.set_title('T_RF(ω) deconvolution with Wiener regularisation (ε = 0.10)\n'
             'Weight rolls off where |T_RF| ≲ ε to suppress noise amplification')
ax.legend(fontsize=9); ax.grid(True, alpha=0.35)
plt.tight_layout()
plt.savefig('Results/T_RF_deconv_weight.png', dpi=200, bbox_inches='tight')
plt.show()

In [ ]:
# ── Compare: linear vs notebook-corrected vs properly calibrated (± T_RF deconv) ──
colors = plt.cm.plasma(np.linspace(0.15, 0.85, len(calibrated)))

# Helper: align scan_data (linear) and corrected (notebook-corrected) to calibrated by Q
by_Q_linear     = {s['Q']: s for s in scan_data}
by_Q_notebook   = {r['Q']: r for r in corrected}

fig = plt.figure(figsize=(16, 10))
gs  = fig.add_gridspec(2, 3, hspace=0.35, wspace=0.30)

# ── Panel 1: Δφ waveforms, three methods overlaid for 10 nC ──────────────
ax = fig.add_subplot(gs[0, 0])
r_cal = calibrated[-1]
r_nb  = by_Q_notebook[r_cal['Q']]
r_lin = by_Q_linear[r_cal['Q']]
ax.plot(r_lin['delay_ps'], r_lin['delta_phi']*1e3, 'C7-',  lw=1.0, alpha=0.7,
        label='Linear  V_sig/V_car  (over-est. ~2×)')
ax.plot(r_nb ['delay_ps'], r_nb ['dph_cor']*1e3,  'C0--', lw=1.2, alpha=0.8,
        label="Notebook 'corrected' (self-fit V_max)")
ax.plot(r_cal['delay_ps'], r_cal['dphi']*1e3,     'C3-',  lw=1.8,
        label='Calibrated  (fitted V_0, φ_bias)')
ax.axhline(0, color='k', lw=0.6, ls=':', alpha=0.4)
ax.set_xlabel('Phase delay (ps)'); ax.set_ylabel('Δφ (mrad)')
ax.set_title(f'10 nC scan — three Δφ-extraction methods')
ax.legend(fontsize=8); ax.grid(True, alpha=0.35)

# ── Panel 2: E_THz with vs without T_RF deconvolution ─────────────────────
ax = fig.add_subplot(gs[0, 1])
ax.plot(r_cal['delay_ps'], r_cal['E_no_deconv']/1e6, 'C0-', lw=1.5, alpha=0.7,
        label='Apparent (T_RF=1 assumed)')
ax.plot(r_cal['delay_ps'], r_cal['E_deconv']/1e6,    'C3-', lw=1.8,
        label='T_RF-deconvolved')
ax.axhline(0, color='k', lw=0.6, ls=':', alpha=0.4)
ax.set_xlabel('Phase delay (ps)'); ax.set_ylabel('E_THz (MV/m)')
ax.set_title('10 nC — T_RF(ω) deconvolution effect')
ax.legend(fontsize=9); ax.grid(True, alpha=0.35)

# ── Panel 3: peak-|E_THz| vs charge, all three methods ────────────────────
ax = fig.add_subplot(gs[0, 2])
Q_arr     = np.array([r['Q'] for r in calibrated])
E_lin_pk  = np.array([np.nanmax(np.abs(by_Q_linear  [Q]['E_THz']))   for Q in Q_arr])
E_nb_pk   = np.array([np.nanmax(np.abs(by_Q_notebook[Q]['E_cor']))   for Q in Q_arr])
E_cal_pk  = np.array([np.nanmax(np.abs(r['E_no_deconv']))            for r in calibrated])
E_dec_pk  = np.array([np.nanmax(np.abs(r['E_deconv'][~np.isnan(r['E_deconv'])])) for r in calibrated])

ax.plot(Q_arr, E_lin_pk/1e6, 'C7s-', lw=1, ms=6, alpha=0.7, label='Linear (orig)')
ax.plot(Q_arr, E_nb_pk /1e6, 'C0o-', lw=1, ms=6, alpha=0.7, label="Notebook 'corrected'")
ax.plot(Q_arr, E_cal_pk/1e6, 'C2^-', lw=1.5, ms=8, label='Calibrated (no deconv)')
ax.plot(Q_arr, E_dec_pk/1e6, 'C3v-', lw=1.5, ms=8, label='Calibrated + T_RF deconv')
# Origin-forced linear fit on calibrated+deconv
slope_dec = (Q_arr*E_dec_pk).sum() / (Q_arr**2).sum()
Q_fit = np.linspace(0, Q_arr.max()*1.05, 100)
ax.plot(Q_fit, slope_dec*Q_fit/1e6, 'C3:', lw=1, alpha=0.7,
        label=f'fit: {slope_dec/1e3:.1f} kV/m/nC')
ax.set_xlabel('Bunch charge (nC)'); ax.set_ylabel('Peak |E_THz| (MV/m)')
ax.set_title('Peak field scaling: 4 methods compared')
ax.legend(fontsize=8); ax.grid(True, alpha=0.35)
ax.set_xlim(left=0); ax.set_ylim(bottom=0)

# ── Panel 4: E_THz(t) waveforms for all charges, calibrated + deconv ──────
ax = fig.add_subplot(gs[1, 0])
for r, col in zip(calibrated, colors):
    ax.plot(r['delay_ps'], r['E_deconv']/1e6, color=col, lw=1.3,
            label=f"{r['Q']:.2g} nC")
ax.axhline(0, color='k', lw=0.6, ls=':', alpha=0.4)
ax.set_xlabel('Phase delay (ps)'); ax.set_ylabel('E_THz (MV/m)  [T_RF-deconv]')
ax.set_title('Calibrated + T_RF-deconvolved E-field waveforms')
ax.legend(title='Charge', fontsize=7, ncol=2); ax.grid(True, alpha=0.35)

# ── Panel 5: per-scan t_probe sample (verify per-scan finding) ────────────
ax = fig.add_subplot(gs[1, 1])
ax.scatter(Q_arr, [r['t_probe_ns'] for r in calibrated], s=70, color='C3', zorder=5)
ax.axhline(np.mean([r['t_probe_ns'] for r in calibrated]), color='C7', ls='--', lw=1,
           label=f'mean = {np.mean([r["t_probe_ns"] for r in calibrated]):.2f} ns')
ax.set_xlabel('Bunch charge (nC)'); ax.set_ylabel('t_probe (ns)')
ax.set_title('Per-scan t_probe (was previously frozen at 10 nC value)')
ax.legend(fontsize=9); ax.grid(True, alpha=0.35)

# ── Panel 6: spectra of E_deconv vs E_no_deconv at 10 nC ──────────────────
ax = fig.add_subplot(gs[1, 2])
r10 = calibrated[-1]
E_nd = r10['E_no_deconv']; E_dc_clean = np.where(np.isnan(r10['E_deconv']), 0, r10['E_deconv'])
N = len(E_nd); dt_s = (r10['delay_ps'][-1]-r10['delay_ps'][0])/(N-1)*1e-12
fr_GHz = np.fft.rfftfreq(N, d=dt_s)/1e9
w = np.hanning(N)
S_nd  = np.abs(np.fft.rfft(E_nd * w))
S_dc  = np.abs(np.fft.rfft(E_dc_clean * w))
ax.semilogy(fr_GHz, S_nd / S_nd.max(), 'C0-', lw=1.5, alpha=0.7, label='Apparent (T_RF=1)')
ax.semilogy(fr_GHz, S_dc / S_dc.max(), 'C3-', lw=1.5, label='T_RF-deconvolved')
ax.set_xlabel('Frequency (GHz)'); ax.set_ylabel('|Ẽ_THz(ω)|  (norm.)')
ax.set_title('10 nC spectrum: high-freq content recovered by deconv')
ax.set_xlim(0, 600); ax.set_ylim(1e-3, 2)
ax.legend(fontsize=9); ax.grid(True, alpha=0.35, which='both')

plt.suptitle('Three-method comparison: Δφ extraction and E_THz reconstruction',
             fontsize=13, y=0.995)
plt.savefig('Results/three_method_comparison.png', dpi=200, bbox_inches='tight')
plt.show()

# ── Summary table ─────────────────────────────────────────────────────────
print('\n' + '='*100)
print(f"{'Q':>5} | {'Δφ_lin':>9} | {'Δφ_nb':>9} | {'Δφ_cal+':>9} | {'Δφ_cal-':>9} | "
      f"{'E_lin':>8} | {'E_nb':>8} | {'E_cal':>8} | {'E_dec':>8}  (peak |·|)")
print(f"{'nC':>5} | {'mrad':>9} | {'mrad':>9} | {'mrad':>9} | {'mrad':>9} | "
      f"{'kV/m':>8} | {'kV/m':>8} | {'kV/m':>8} | {'kV/m':>8}")
print('-'*100)
for r in calibrated:
    Q = r['Q']
    rl = by_Q_linear[Q]; rn = by_Q_notebook[Q]
    pk_lin = np.nanmax(np.abs(rl['delta_phi']))*1e3
    pk_nb  = np.nanmax(np.abs(rn['dph_cor']))*1e3
    pk_p   = np.nanmax(r['dphi'][r['dphi']>0])*1e3 if np.any(r['dphi']>0) else float('nan')
    pk_n   = np.abs(np.nanmin(r['dphi'][r['dphi']<0]))*1e3 if np.any(r['dphi']<0) else float('nan')
    El     = np.nanmax(np.abs(rl['E_THz']))/1e3
    En     = np.nanmax(np.abs(rn['E_cor']))/1e3
    Ec     = np.nanmax(np.abs(r['E_no_deconv']))/1e3
    Ed     = np.nanmax(np.abs(r['E_deconv'][~np.isnan(r['E_deconv'])]))/1e3
    print(f"{Q:5.2f} | {pk_lin:9.0f} | {pk_nb:9.0f} | {pk_p:9.0f} | {pk_n:9.0f} | "
          f"{El:8.0f} | {En:8.0f} | {Ec:8.0f} | {Ed:8.0f}")
print('='*100)
print(f'\nBeamline factor (calibrated + T_RF-deconv): {slope_dec/1e3:.1f} kV/m per nC')
print(f'  vs original linear estimate: ~217 kV/m per nC')
print(f'  vs notebook-corrected:       ~160 kV/m per nC')

In [6]:
def find_zero_crossings(x, y):
    """Linear interpolation of zero crossings of y(x)."""
    idx = np.where(np.diff(np.sign(y)))[0]
    crossings = []
    for i in idx:
        x0, x1 = x[i], x[i + 1]
        y0, y1 = y[i], y[i + 1]
        crossings.append(x0 - y0 * (x1 - x0) / (y1 - y0))
    return np.array(crossings)

colors = plt.cm.plasma(np.linspace(0.15, 0.85, len(scan_data)))

fig, axes = plt.subplots(1, 2, figsize=(14, 5))

# --- Left: delta_phi waveforms with zero crossings marked ---
ax = axes[0]
print(f"{'Scan':>8}  {'Zero crossings (ps)':>40}  {'Δφ at peak (mrad)':>18}")
print('-' * 75)
for s, col in zip(scan_data, colors):
    dp  = s['delay_ps']
    phi = s['delta_phi']
    zc  = find_zero_crossings(dp, phi)

    ax.plot(dp, phi * 1e3, color=col, lw=1.5, label=f"{s['Q']:.2g} nC")
    ax.scatter(zc, np.zeros_like(zc), color=col, s=40, zorder=5, marker='|', linewidths=2)

    zc_str = '  '.join(f'{z:.1f}' for z in zc)
    print(f"{s['Q']:>6.2f} nC  [{zc_str}]  {s['peak_phi']*1e3:>14.1f} mrad")

ax.axhline(0, color='k', lw=0.8, ls='--', alpha=0.5)
ax.set_xlabel('Phase delay (ps)', fontsize=11)
ax.set_ylabel('Δφ (mrad)', fontsize=11)
ax.set_title('Phase shift waveforms\nTick marks = quadrature zero crossings (Δφ = 0)')
ax.legend(title='Charge', fontsize=8); ax.grid(True, alpha=0.35)

# --- Right: zoom on the 10 nC scan to show crossings clearly ---
ax2 = axes[1]
s10 = scan_data[-1]
dp, phi = s10['delay_ps'], s10['delta_phi']
zc = find_zero_crossings(dp, phi)
ax2.plot(dp, phi * 1e3, color=colors[-1], lw=2, label=f"{s10['Q']:.0f} nC")
ax2.axhline(0, color='k', lw=0.8, ls='--', alpha=0.5)
for i, z in enumerate(zc):
    ax2.axvline(z, color='C3', ls=':', lw=1.2, alpha=0.8,
                label='Zero crossings' if i == 0 else '')
    ax2.scatter([z], [0], color='C3', s=60, zorder=6)
    ax2.annotate(f'{z:.1f} ps', (z, phi.max()*1e3*0.1 + i*phi.max()*1e3*0.12),
                 fontsize=8, color='C3', ha='center')

# Shade ±5 mrad band around zero to highlight the linear regime
ax2.axhspan(-5, 5, color='grey', alpha=0.12, label='|Δφ| < 5 mrad (linear regime)')
ax2.set_xlabel('Phase delay (ps)', fontsize=11)
ax2.set_ylabel('Δφ (mrad)', fontsize=11)
ax2.set_title(f'10 nC scan — quadrature points\n(MZI most sensitive at Δφ = 0)')
ax2.legend(fontsize=8); ax2.grid(True, alpha=0.35)

plt.tight_layout()
plt.savefig('Results/quadrature_zero_crossings.png', dpi=200, bbox_inches='tight')
plt.show()


    Scan                       Zero crossings (ps)   Δφ at peak (mrad)
---------------------------------------------------------------------------
  0.25 nC  [0.1  3.2  20.7  23.7  24.2  31.7  41.4  47.5  49.8  52.9  58.7  68.6  70.3  72.8  74.7  78.1  83.5  85.5  87.1  94.5  99.5  105.5  112.0  113.1  114.9  117.7  118.3  119.1  120.6  123.1  124.2  125.3  126.6  128.0  129.6  131.3  132.5  133.7  134.8]            71.2 mrad
  0.50 nC  [0.4  2.9  10.3  11.9  14.9  24.2  34.1  43.7  50.0  52.5  55.6  62.6  69.3  70.7  74.9  75.7  78.0  84.4  86.9  88.2  96.5  101.8  107.7  118.9  121.4  123.6  124.7  125.0  132.5  133.4  134.7  135.4]           141.3 mrad
  1.00 nC  [4.1  6.3  13.9  17.0  17.1  27.5  36.6  38.3  38.6  47.5  54.4  56.4  59.4  69.4  73.3  75.1  75.8  77.7  81.5  82.3  84.2  91.1  95.7  96.8  102.8  109.2  116.1  123.1  124.3  127.3  130.5  131.4  132.7  133.9]           257.4 mrad
  4.00 nC  [0.6  10.1  13.3  22.8  34.9  46.0  54.1  65.8  74.1  78.6  79.1  79.6  81.4  87

/tmp/ipykernel_45695/459907932.py:59: UserWarning: FigureCanvasAgg is non-interactive, and thus cannot be shown
  plt.show()


In [7]:
# ── Plot: E_THz waveform vs delay for all charges ────────────────────────────
colors = plt.cm.plasma(np.linspace(0.15, 0.85, len(scan_data)))

fig, axes = plt.subplots(1, 2, figsize=(13, 5))

ax = axes[0]
for s, col in zip(scan_data, colors):
    ax.plot(s['delay_ps'], s['E_THz']/1e6, color=col, lw=1.5, label=f"{s['Q']:.2g} nC")
ax.axhline(0, color='k', lw=0.6, ls='--', alpha=0.4)
ax.set_xlabel('Phase delay (ps)', fontsize=11)
ax.set_ylabel('E_THz (MV/m)', fontsize=11)
ax.set_title('THz Electric Field Waveform E(t)')
ax.legend(title='Charge', fontsize=8); ax.grid(True, alpha=0.35)

ax = axes[1]
Q_vals  = np.array([s['Q']     for s in scan_data])
E_peaks = np.array([s['peak_E'] for s in scan_data])
# Origin-forced fit
slope_E = (Q_vals * E_peaks).sum() / (Q_vals**2).sum()
Q_fit   = np.linspace(0, Q_vals.max()*1.05, 200)
ax.scatter(Q_vals, E_peaks/1e6, color=colors, s=70, zorder=5, label='Data')
ax.plot(Q_fit, slope_E*Q_fit/1e6, 'C3-', lw=2,
        label=f'Origin fit: {slope_E/1e6*1e3:.1f} kV/m per nC')
ax.set_xlabel('Nominal charge (nC)', fontsize=11)
ax.set_ylabel('Peak |E_THz| (MV/m)', fontsize=11)
ax.set_title('Peak THz Field vs Bunch Charge')
ax.legend(fontsize=9); ax.grid(True, alpha=0.35)
ax.set_xlim(left=0); ax.set_ylim(bottom=0)

plt.tight_layout()
plt.savefig('Results/E_THz_waveform_and_scaling.png', dpi=200, bbox_inches='tight')
plt.show()
print(f'Beamline factor: {slope_E/1e3:.1f} kV/m per nC')

Beamline factor: 216.8 kV/m per nC


/tmp/ipykernel_45695/2637243948.py:32: UserWarning: FigureCanvasAgg is non-interactive, and thus cannot be shown
  plt.show()


In [8]:
# ── THz frequency spectrum via FFT of EO scan ────────────────────────────────
fig, (ax_t, ax_f) = plt.subplots(1, 2, figsize=(13, 5))

for s, col in zip(scan_data, colors):
    dp   = s['delay_ps']
    E    = s['E_THz']
    n    = len(E)
    dt_ps = (dp[-1] - dp[0]) / (n-1)
    win  = np.hanning(n)
    F    = np.fft.rfft(E * win)
    freq = np.fft.rfftfreq(n, d=dt_ps*1e-12) / 1e12  # THz
    spec = np.abs(F)

    ax_t.plot(dp, E/1e6, color=col, lw=1.5, label=f"{s['Q']:.2g} nC")
    ax_f.plot(freq*1e3, spec/spec.max(), color=col, lw=1.5, label=f"{s['Q']:.2g} nC")

ax_t.axhline(0, color='k', lw=0.6, ls='--', alpha=0.4)
ax_t.set_xlabel('Phase delay (ps)', fontsize=11); ax_t.set_ylabel('E_THz (MV/m)', fontsize=11)
ax_t.set_title('THz waveform (time domain)'); ax_t.legend(title='Charge', fontsize=8); ax_t.grid(True, alpha=0.35)

ax_f.set_xlabel('Frequency (GHz)', fontsize=11); ax_f.set_ylabel('Normalised amplitude', fontsize=11)
dp0 = scan_data[0]['delay_ps']
n0  = len(dp0); dt0 = (dp0[-1]-dp0[0])/(n0-1)
ax_f.set_title(f'THz spectrum (Nyquist={0.5/dt0*1e3:.0f} GHz, Δf={1e3/(n0*dt0):.1f} GHz)')
ax_f.set_xlim(0, 600); ax_f.legend(title='Charge', fontsize=8); ax_f.grid(True, alpha=0.35)

plt.tight_layout()
plt.savefig('Results/THz_spectrum_from_EO_scan.png', dpi=200, bbox_inches='tight')
plt.show()

/tmp/ipykernel_45695/1439742463.py:29: UserWarning: FigureCanvasAgg is non-interactive, and thus cannot be shown
  plt.show()


In [9]:
# ── Summary table ─────────────────────────────────────────────────────────────
print('=' * 85)
print(f"{'Q_nom':>6} | {'V_max (mV)':>10} | {'peak Δφ (mrad)':>14} | {'peak E (MV/m)':>13} | {'E/nC (kV/m/nC)':>14}")
print('-' * 85)
for s in scan_data:
    print(f"{s['Q']:6.2f} | {s['V_max']*1e3:10.1f} | {s['peak_phi']*1e3:14.2f} | "
          f"{s['peak_E']/1e6:13.3f} | {s['peak_E']/s['Q']/1e3:14.1f}")
print('=' * 85)
print(f'\nPhysical parameters (Nature paper):')
print(f'  λ     = {lam*1e9:.0f} nm')
print(f'  n_e   = {n_e}')
print(f'  r₃₃   = {r33*1e12:.1f} pm/V')
print(f'  l     = {l*1e6:.0f} μm')
print(f'  E_π   = {E_pi/1e6:.3f} MV/m')
print(f'\nBeamline geometry factor: {slope_E/1e3:.1f} kV/m per nC at crystal face')

 Q_nom | V_max (mV) | peak Δφ (mrad) | peak E (MV/m) | E/nC (kV/m/nC)
-------------------------------------------------------------------------------------
  0.25 |      174.0 |          71.17 |         0.095 |          381.1
  0.50 |      179.2 |         141.33 |         0.189 |          378.4
  1.00 |      175.2 |         257.40 |         0.345 |          344.6
  4.00 |      172.2 |         745.65 |         0.998 |          249.6
  5.00 |      170.0 |         985.04 |         1.319 |          263.8
  7.00 |      174.0 |        1185.87 |         1.588 |          226.8
 10.00 |      168.0 |        1442.60 |         1.931 |          193.1

Physical parameters (Nature paper):
  λ     = 1550 nm
  n_e   = 2.15
  r₃₃   = 30.9 pm/V
  l     = 600 μm
  E_π   = 4.206 MV/m

Beamline geometry factor: 216.8 kV/m per nC at crystal face
